In [8]:
import sys
sys.path.insert(0, '/Users/shelleygoel/Code/01_statistical_mod_blog/anomaly_detection')

from pathlib import Path
import numpy as np
import pandas as pd
from core.dataset import TimeSeriesDataset
from core.evaluation import Evaluation
from core.models import EuclideanDistModel, Catch22MPModel, UniformWeighter, FeatureWeighter
from core.feature_transformer import FeatureTransformer, FeatureCategory
from core.viz import plot_cases

# Load HVAC Data

In [5]:
hvac_df = pd.read_parquet(Path("../datasets/hvac_anomalies_v021226.parquet"))

hvac_df = hvac_df.copy()
hvac_df["TmpRet"] = (
    hvac_df.groupby(["container_id", "unit"])["TmpRet"]
    .transform(lambda x: x.rolling(window=10, min_periods=1).mean())
)

In [6]:

col_map = {
    'entity': 'container_id',
    'time': 'timestamp_et',
    'value_cols': ['TmpRet'],
    'label': 'anomaly',
    'label_type': 'anomaly_type',
    'sub_entity': 'unit',
}
ds = TimeSeriesDataset(hvac_df, col_map)                                                                                                                                                                                                                                                
 

In [7]:
 
sampled_entities = ds.sample_entities(n_cases=50, label_type="frequency", random_state=42)              
normal_entities = ds.sample_entities(n_cases=100, label_type="normal", random_state=42)   


In [9]:
train_entities = np.concatenate([sampled_entities, normal_entities])

In [10]:
           
ds_small = TimeSeriesDataset(hvac_df[hvac_df['container_id'].isin(train_entities)], col_map)                                                                                                                                                                                          
 

In [23]:
                                                                                                                                                                                                                                                                                         
# 3. Feature transform                                                                                                                                                                                                                                                                  
ft = FeatureTransformer(raw_data_columns=['TmpRet'], window_size=12 * 60, stride=60, n_jobs=8)                                                                                                                                                                                                                
feat_ds = ft.transform(ds_small) #categories=[FeatureCategory.C22_DIFF])                                                                                                                                                                                                                 
             

FeatureTransformer: 100%|██████████| 121/121 [00:37<00:00,  3.27it/s]


In [24]:
stride = ft.stride                # 45
exclude_zone = 1440 // stride     # 32
print(f"{stride=}, {exclude_zone=}")

stride=60, exclude_zone=24


In [25]:

calc_features = sum([list(v) for v in ft.feature_map.values()], [])
import re
class CustomWeighter(FeatureWeighter):
    def compute_weights(self, preferred_features: list) -> dict:
        return dict([(feat, 1) for feat in preferred_features])
pattern = re.compile(r'CO_f1ecac|SB_BinaryStats_diff')
pref_features = [col for col in calc_features if pattern.search(col)]
print(pref_features)
cw = CustomWeighter()
custom_wei = cw.compute_weights(preferred_features=pref_features)

['TmpRet__0__CO_f1ecac', 'TmpRet__0__SB_BinaryStats_diff_longstretch0', 'TmpRet__1__CO_f1ecac', 'TmpRet__1__SB_BinaryStats_diff_longstretch0', 'TmpRet__2__CO_f1ecac', 'TmpRet__2__SB_BinaryStats_diff_longstretch0', 'TmpRet__0_1__CO_f1ecac', 'TmpRet__0_1__SB_BinaryStats_diff_longstretch0', 'TmpRet__0_2__CO_f1ecac', 'TmpRet__0_2__SB_BinaryStats_diff_longstretch0', 'TmpRet__1_2__CO_f1ecac', 'TmpRet__1_2__SB_BinaryStats_diff_longstretch0']


In [29]:
calc_features

['TmpRet__0__DN_HistogramMode_5',
 'TmpRet__0__DN_HistogramMode_10',
 'TmpRet__0__CO_f1ecac',
 'TmpRet__0__CO_FirstMin_ac',
 'TmpRet__0__CO_HistogramAMI_even_2_5',
 'TmpRet__0__CO_trev_1_num',
 'TmpRet__0__MD_hrv_classic_pnn40',
 'TmpRet__0__SB_BinaryStats_mean_longstretch1',
 'TmpRet__0__SB_TransitionMatrix_3ac_sumdiagcov',
 'TmpRet__0__PD_PeriodicityWang_th0_01',
 'TmpRet__0__CO_Embed2_Dist_tau_d_expfit_meandiff',
 'TmpRet__0__IN_AutoMutualInfoStats_40_gaussian_fmmi',
 'TmpRet__0__FC_LocalSimple_mean1_tauresrat',
 'TmpRet__0__DN_OutlierInclude_p_001_mdrmd',
 'TmpRet__0__DN_OutlierInclude_n_001_mdrmd',
 'TmpRet__0__SP_Summaries_welch_rect_area_5_1',
 'TmpRet__0__SB_BinaryStats_diff_longstretch0',
 'TmpRet__0__SB_MotifThree_quantile_hh',
 'TmpRet__0__SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1',
 'TmpRet__0__SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1',
 'TmpRet__0__SP_Summaries_welch_rect_centroid',
 'TmpRet__0__FC_LocalSimple_mean3_stderr',
 'TmpRet__1__DN_HistogramMode_5',
 'TmpRet__1__D

In [32]:
                                                                                                                                                                                                                                                                             
# 4. Fit MP once                                                                                                                                                                                                                                                                        
c22mp = Catch22MPModel(                                                                                                                                                                                         
      exclude_zone=exclude_zone,                                                                                                                                                                                    
      early_abandon=False,       # brute force — fast enough                                                                                                                                              
  )   
mp_ds, debug = c22mp.fit_profile(feat_ds, weights=UniformWeighter().compute_weights(feature_names=calc_features))                                                                                                                                                                                                                           
# c22mp = Catch22MPModel(window_size=24*60,exclude_zone=24*60, early_abandon=False, score_day_agg_stat='max')                                                                                                                                                                                               
# mp2_ds, debug = c22mp.fit_profile(feat_ds, weights=custom_wei)                                                                                                                                                                                                                           
                                            
                

Catch22MP fit_profile: 100%|██████████| 121/121 [00:00<00:00, 806.75it/s]


In [ ]:
feat_cfg = feat_ds.to_plot_cfg()                                                                                                                                                                                                                                                                           
feat_cfg.value_cols = sorted([col for col in feat_ds.to_plot_cfg().value_cols if col.find("0_2") != -1] )  
skip = True
if not skip:                                                                                                                                                                                                                                                                   
  figs = plot_cases(
        # [ds_small.to_plot_cfg(),
         [mp_ds.to_plot_cfg(), feat_cfg],
        sample_from=ds_small,
        n_cases=3,
        # entity_ids=[246],
        label_type="frequency",
        labels_from=ds_small.to_plot_cfg(),
        random_state=42,
    )  
  for fig in figs: fig.show()                   
                                                                                                                                                                                                                                                                                        
# 7. Evaluate                                                                                                  
# ev = Evaluation(level='day')                                                                                                                                                                                                                                                            
# print(ev.metrics_table(day_scores, ds_small))

In [15]:
ds_small.anomaly_summary()

,label_type,entity_count
0,normal,88
1,frequency,24
2,lag,6
3,amplitude,3


In [33]:
ev = Evaluation(level="day")
c22_day_scores = c22mp.score(mp_ds, level="day", day_agg_stat="p90")

fig = ev.plot_pr_curves_compared(
    {"Catch22MP": c22_day_scores},                                                                                                                                                                                                                                           
    ds_small,                                                                                                                                                                                                                                                                                              
)                                                                                                                                                                                                                                                                                                          
fig.show() 

In [22]:
ev = Evaluation(level="day")
c22_day_scores = c22mp.score(mp_ds, level="day", day_agg_stat="p90")

fig = ev.plot_pr_curves_compared(
    {"Catch22MP": c22_day_scores},                                                                                                                                                                                                                                           
    ds_small,                                                                                                                                                                                                                                                                                              
)                                                                                                                                                                                                                                                                                                          
fig.show() 

In [16]:

ev = Evaluation(level="day")
c22_day_scores = c22mp.score(mp_ds, level="day", day_agg_stat="p90")

fig = ev.plot_pr_curves_compared(
    {"Catch22MP": c22_day_scores},                                                                                                                                                                                                                                           
    ds_small,                                                                                                                                                                                                                                                                                              
)                                                                                                                                                                                                                                                                                                          
fig.show() 

In [17]:

eucl = EuclideanDistModel(feature_col="TmpRet", smooth_window=1, dist_window=60, strategy="mad")
eucl_day_scores = eucl.score_anomalies(ds_small, level="day")

# Catch22 MP scoring (already have day_scores from earlier)
# day_scores = c22mp.score(mp_ds, level="day")

# Compare
ev = Evaluation(level="day")

# Metrics side-by-side
print(ev.compare(
    {"Catch22MP": c22_day_scores, "EuclideanDist": eucl_day_scores},
    ds_small,
))

# Overlaid PR curves, one subplot per anomaly type
fig = ev.plot_pr_curves_compared(
    {"Catch22MP": c22_day_scores, "EuclideanDist": eucl_day_scores},                                                                                                                                                                                                                                           
    ds_small,                                                                                                                                                                                                                                                                                              
)                                                                                                                                                                                                                                                                                                          
fig.show()                                       

  anomaly_type  Catch22MP_auc_pr  Catch22MP_auc_roc  EuclideanDist_auc_pr  \
0    frequency          0.440237           0.816248              0.777467   
1          lag          0.096821           0.730779              0.396550   
2    amplitude          0.265433           0.665101              0.008365   
3      overall          0.472179           0.788685              0.749341   

   EuclideanDist_auc_roc  
0               0.934079  
1               0.942003  
2               0.343542  
3               0.892504  
